In [15]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder
import joblib

In [16]:
# Configuration
PROCESSED_PATH = "../processed_data"
MASTER_PATH = "../data/master"
MODELS_PATH = "../models"

os.makedirs(MASTER_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

In [17]:
# Load processed datasets
print("\n Loading processed datasets...")
arrivals_country = pd.read_csv(f"{PROCESSED_PATH}/Tourist_Arrivals_By_Country_Month_2019_2024.csv")
top10_markets = pd.read_csv(f"{PROCESSED_PATH}/Top10_Source_Markets_2019_2024.csv")
accommodation = pd.read_csv(f"{PROCESSED_PATH}/Accommodation_By_Province_2019_2024.csv")
attractions = pd.read_csv(f"{PROCESSED_PATH}/Attraction_Revenue_Visitors_2019_2024.csv")

# Aggregate accommodation data
accommodation_yearly = accommodation.groupby('Year').agg({
    'Number of Rooms': ['sum', 'mean', 'std']
}).round(2)
accommodation_yearly.columns = ['Total_Rooms', 'Avg_Rooms_per_Province', 'Std_Rooms_per_Province']
accommodation_yearly = accommodation_yearly.reset_index()

# Top provinces
province_totals = accommodation.groupby('Province')['Number of Rooms'].sum().sort_values(ascending=False)
top_provinces = province_totals.head(5).index.tolist()
print(f"  Top 5 provinces: {top_provinces}")

accommodation_province = accommodation[accommodation['Province'].isin(top_provinces)].pivot_table(
    index='Year', 
    columns='Province', 
    values='Number of Rooms',
    fill_value=0
).reset_index()
accommodation_province.columns = ['Year'] + [f'Rooms_{col.replace(" ", "_")}' for col in accommodation_province.columns[1:]]

# Aggregate attractions data
attractions_yearly = attractions.groupby('Year').agg({
    'Foreign Number': 'sum',
    'Local Number': 'sum',
    'Foreign Income': 'sum',
    'Local Income': 'sum'
}).round(2).reset_index()
attractions_yearly.columns = ['Year', 
                              'Total_Foreign_Visitors_Attractions',
                              'Total_Local_Visitors_Attractions',
                              'Total_Foreign_Income', 
                              'Total_Local_Income']

# Process top10 markets
all_countries = arrivals_country['Country'].unique()

country_years = []
for year in [2019, 2020, 2021, 2022, 2023, 2024]:
    for country in all_countries:
        country_years.append({'Country': country, 'Year': year})

top10_all = pd.DataFrame(country_years)
top10_all = top10_all.merge(
    top10_markets[['Country', 'Year', 'Share']], 
    on=['Country', 'Year'], 
    how='left'
)

top10_all['Is_Top10'] = (~top10_all['Share'].isna()).astype(int)
top10_all['Market_Share'] = top10_all['Share'].fillna(0)

country_top10_years = top10_all.groupby('Country')['Is_Top10'].sum().reset_index()
country_top10_years.columns = ['Country', 'Top10_Years']
top10_all = top10_all.merge(country_top10_years, on='Country', how='left')
top10_all['Is_Consistent_Top10'] = (top10_all['Top10_Years'] >= 3).astype(int)

# Process arrivals by country-month
month_names = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

master = arrivals_country.melt(
    id_vars=['Country', 'Year'],
    value_vars=month_names,
    var_name='Month',
    value_name='Tourist_Arrivals'
)

month_to_num = {month: i+1 for i, month in enumerate(month_names)}
master['Month_Num'] = master['Month'].map(month_to_num)
master['Month_Sin'] = np.sin(2 * np.pi * master['Month_Num'] / 12)
master['Month_Cos'] = np.cos(2 * np.pi * master['Month_Num'] / 12)

def get_quarter(month_num):
    if month_num <= 3:
        return 'Q1'
    elif month_num <= 6:
        return 'Q2'
    elif month_num <= 9:
        return 'Q3'
    else:
        return 'Q4'

master['Quarter'] = master['Month_Num'].apply(get_quarter)


 Loading processed datasets...
  Top 5 provinces: ['Western Province', 'Southern Province', 'Central Province', 'North Central Province', 'Eastern Province']


In [18]:
# ============================================
# FEATURE ENGINEERING
# ============================================
print("\n Creating enhanced features...")

# Sort for time series operations
master = master.sort_values(['Country', 'Year', 'Month_Num'])

# COVID indicators
print("  • COVID indicators")
master['Is_COVID_Year'] = master['Year'].isin([2020, 2021]).astype(int)
master['Is_Post_COVID'] = master['Year'].isin([2022, 2023, 2024]).astype(int)
master['COVID_Severity'] = 0
master.loc[(master['Year'] == 2020) & (master['Month_Num'] >= 3), 'COVID_Severity'] = 1
master.loc[(master['Year'] == 2020) & (master['Month_Num'] >= 4), 'COVID_Severity'] = 2
master.loc[(master['Year'] == 2020) & (master['Month_Num'] >= 5), 'COVID_Severity'] = 3
master.loc[(master['Year'] == 2020), 'COVID_Severity'] = 3
master.loc[(master['Year'] == 2021), 'COVID_Severity'] = 2
master.loc[(master['Year'] == 2022), 'COVID_Severity'] = 1

# Country-specific lag features
print("  • Country-specific lag features")
for lag in [1, 2, 3, 6, 12]:
    master[f'Lag_{lag}_Month'] = master.groupby('Country')['Tourist_Arrivals'].shift(lag)

# Rolling statistics
print("  • Rolling statistics")
for window in [3, 6, 12]:
    master[f'Rolling_Mean_{window}'] = master.groupby('Country')['Tourist_Arrivals'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )
    master[f'Rolling_Std_{window}'] = master.groupby('Country')['Tourist_Arrivals'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).std()
    )

# Seasonal features
print("  • Seasonal features")
master['Prev_Year_Same_Month'] = master.groupby(['Country', 'Month'])['Tourist_Arrivals'].shift(12)
master['Prev_Year_Same_Month_Ratio'] = master['Tourist_Arrivals'] / (master['Prev_Year_Same_Month'] + 1).clip(lower=1)

# Growth rate features
print("  • Growth rate features")
master['Growth_3M'] = (master['Tourist_Arrivals'] - master['Rolling_Mean_3']) / (master['Rolling_Mean_3'] + 1).clip(lower=1)
master['Growth_12M'] = (master['Tourist_Arrivals'] - master['Rolling_Mean_12']) / (master['Rolling_Mean_12'] + 1).clip(lower=1)

# Merge with other datasets
master = master.merge(accommodation_yearly, on='Year', how='left')
master = master.merge(accommodation_province, on='Year', how='left')
master = master.merge(attractions_yearly, on='Year', how='left')
master = master.merge(top10_all[['Country', 'Year', 'Is_Top10', 'Market_Share', 'Is_Consistent_Top10']], 
                      on=['Country', 'Year'], how='left')

# Interaction features
print("  • Interaction features")
master['COVID_Season_Interaction'] = master['COVID_Severity'] * master['Month_Sin']
master['Top10_COVID_Effect'] = master['Is_Top10'] * master['COVID_Severity']

# Log transformations
print("  • Log transformations")
skewed_features = ['Total_Rooms', 'Total_Foreign_Income', 'Total_Local_Income']
for feat in skewed_features:
    if feat in master.columns:
        master[f'{feat}_log'] = np.log1p(master[feat].clip(lower=0))

# Country-level features
print("  • Country-level features")
country_total = master.groupby('Country')['Tourist_Arrivals'].sum().reset_index()
country_total.columns = ['Country', 'Total_Country_Arrivals']
country_monthly_avg = master.groupby('Country')['Tourist_Arrivals'].mean().reset_index()
country_monthly_avg.columns = ['Country', 'Avg_Country_Arrivals']
peak_month = master.loc[master.groupby('Country')['Tourist_Arrivals'].idxmax(), ['Country', 'Month']].reset_index(drop=True)
peak_month.columns = ['Country', 'Peak_Month']

master = master.merge(country_total, on='Country', how='left')
master = master.merge(country_monthly_avg, on='Country', how='left')
master = master.merge(peak_month, on='Country', how='left')
master['Is_Peak_Month'] = (master['Month'] == master['Peak_Month']).astype(int)

# Fill NaN values
master = master.fillna(method='ffill').fillna(method='bfill').fillna(0)

# Encode categorical variables
print("  • Encoding categorical variables")
label_encoder = LabelEncoder()
master['Country_Code'] = label_encoder.fit_transform(master['Country'])

quarter_dummies = pd.get_dummies(master['Quarter'], prefix='Quarter')
master = pd.concat([master, quarter_dummies], axis=1)

# Define target
target = 'Tourist_Arrivals'

# Get all numeric features except target
all_features = [col for col in master.columns 
                if col not in ['Country', 'Month', 'Peak_Month', 'Quarter', target]
                and master[col].dtype in ['int64', 'float64']]

print(f"\n Total features created: {len(all_features)}")


 Creating enhanced features...
  • COVID indicators
  • Country-specific lag features
  • Rolling statistics
  • Seasonal features
  • Growth rate features
  • Interaction features
  • Log transformations
  • Country-level features
  • Encoding categorical variables

 Total features created: 46


In [19]:
# Save master dataset
master[all_features + [target]].to_csv(f"{MASTER_PATH}/tourism_master_final.csv", index=False)
joblib.dump(label_encoder, f"{MODELS_PATH}/label_encoder_final.pkl")
joblib.dump(all_features, f"{MODELS_PATH}/feature_list_final.pkl")

print(f"\n Saved: {MASTER_PATH}/tourism_master_final.csv")
print(f"   Shape: {master[all_features + [target]].shape}")
print(f"   Features: {len(all_features)}")


 Saved: ../data/master/tourism_master_final.csv
   Shape: (13632, 47)
   Features: 46
